In [ ]:
import os
os.environ["GROQ_API_KEY"] = "xxxxxxxxxxxxxxxxxxWGdyb3FYUxxczHRErM0CdeLNNEf9pk4K"

## Install libraries

In [10]:
%pip install -U langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface langchain-groq faiss-cpu sentence-transformers youtube-transcript-api python-dotenv

ERROR: Could not find a version that satisfies the requirement langchain.text_splitter (from versions: none)
ERROR: No matching distribution found for langchain.text_splitter


In [10]:
# YouTube Transcript API
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import (
    TranscriptsDisabled,
    NoTranscriptFound,
    VideoUnavailable
)

# Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# HuggingFace Embeddings (Free)
from langchain_huggingface import HuggingFaceEmbeddings

# FAISS Vector Store
from langchain_community.vectorstores import FAISS

# Groq LLM (Free)
from langchain_groq import ChatGroq

# Prompt Template
from langchain_core.prompts import PromptTemplate

# Runnables
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda,
)

# Output Parser
from langchain_core.output_parsers import StrOutputParser

# Environment Variables
from dotenv import load_dotenv

# Standard Libraries
import os
print("All libraries installed successfully!")

All libraries installed successfully!


## Step 1a - Indexing (Document Ingestion)

In [67]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "9g_O05YEKQs"

ytt_api = YouTubeTranscriptApi()

transcript = ytt_api.fetch(
    video_id,
    languages=["hi"]      # Hindi transcript
)

text = " ".join(snippet.text for snippet in transcript)

print(text)

हाउ द हेक? कंट्री विथ 1.4 बिलियन पीपल। डस नॉट हैव 11 गुड प्लेयर्स हु कैन रिप्रेजेंट इन द फीफा वर्ल्ड कप? [संगीत] व्हेन विल वी गेट टू द वर्ल्ड कप? व्हेन यू स्टॉप आस्किंग दिस क्वेश्चन आफ्टर एव्री फोर इयर्स? एंड यू आस्क दिस क्वेश्चन एव्री सिंगल डे, देन वी विल गेट टू द वर्ल्ड कप? हमारे मेसी रोनाल्डो पैदा होकर मर गए हैं। आर सिस्टम इज बिजी किलिंग ऑल द टैलेंट वी हैव। व्हाट आर वी डूइंग रों? हाउ मेनी फुटबॉल स्टेडियम्स डू वी हैव इन द कंट्री? आई डोंट नो हाउ मेनी? जीरो। नाउ व्हेन यू हैव ज़ीरो स्टेडियम्स डेडिकेटेड फॉर फुटबॉल देन हाउ आर यू गोइंग टू गेट देयर सिस्टम में प्रॉब्लम है। अनलेस यू हैव प्रोफेशनल्स रनिंग द स्पोर्ट्स एंड यू डू नॉट हैव पीपल हु आर इक्टेड इदर बिकॉज़ ऑफ़ नेपोटिज्म ओर बिकॉज़ ऑफ़ पावर यू विल नेवर डू वेल इन स्पोर्ट्स। गिव मी योर ऑब्सकल व्हिच सिस्टम क्रिएट्स। मेरिट इज नॉट काउंटेड। व्हाट एल्स ही इज़ लेफ्ट। तेरे पास जेब में कितने पैसे हैं? तू कितनी ब्राइट दे सकता है? या कितनी सिफारिश लगा सकता है? दैट इज वन एडवाइस नोबडी शुड फॉलो प्लेड सेफ नेवर प्ले इट सेफ यू आर इन योर कम्फर्ट ज़ोन यू नेवर गोना 

In [69]:
transcript

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='हाउ द हेक? कंट्री विथ 1.4 बिलियन पीपल।', start=0.08, duration=5.759), FetchedTranscriptSnippet(text='डस नॉट हैव 11 गुड प्लेयर्स हु कैन', start=3.6, duration=5.68), FetchedTranscriptSnippet(text='रिप्रेजेंट इन द फीफा वर्ल्ड कप? [संगीत]', start=5.839, duration=4.88), FetchedTranscriptSnippet(text='व्हेन विल वी गेट टू द वर्ल्ड कप? व्हेन', start=9.28, duration=2.8), FetchedTranscriptSnippet(text='यू स्टॉप आस्किंग दिस क्वेश्चन आफ्टर', start=10.719, duration=2.641), FetchedTranscriptSnippet(text='एव्री फोर इयर्स? एंड यू आस्क दिस', start=12.08, duration=2.719), FetchedTranscriptSnippet(text='क्वेश्चन एव्री सिंगल डे, देन वी विल गेट', start=13.36, duration=3.12), FetchedTranscriptSnippet(text='टू द वर्ल्ड कप? हमारे मेसी रोनाल्डो पैदा', start=14.799, duration=4.161), FetchedTranscriptSnippet(text='होकर मर गए हैं। आर सिस्टम इज बिजी किलिंग', start=16.48, duration=6.36), FetchedTranscriptSnippet(text='ऑल द टैलेंट वी हैव।', start=18.96, dura

## Step 1b - Indexing (Text Splitting)

In [70]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([text])

In [71]:
len(chunks)

199

In [72]:
chunks[100]

Document(metadata={}, page_content='17 राइट नाउ मोरा फॉर मेक्सिको। और यू हैव निको विलियम्स। सी ऑल दीज़ गायज़ हैव स्टार्टेड प्लेइंग एट द एज ऑफ़ 15 एंड 16 विथ द सीनियर्स। आई स्टिल होल्ड द रिकॉर्ड देयर एंड आई स्टिल द होल्ड रिकॉर्ड इन आई लीग फॉर द यंगस्टेस्ट एवर स्कोरर हिमांशु जांगड़ एट 15 16 बिकॉज़ आई प्लेड। नाउ आई कांट प्ले देम देयर। व्हाई? बिकॉज़ दे सेड ओनली पीपल हु आर अबव 18 कैन प्ले। दे कैन ओनली प्ले बिकॉज़ वी कांट टेक गारंटी फॉर देर सेफ्टी। व्हाट आर यू टॉकिंग अबाउट? बाय 18 आई मीन ओ माय गॉड यू डोंट अंडरस्टैंड फुटबॉल। सो दैट मींस स्पोर्ट्स बाय 18 एक्चुअली लॉट ऑफ़ थिंग्स हैव बीन डिसाइडेड ऑलराइट। एववरीथिंग इफ आई टॉकिंग अबाउट द वर्ल्ड लेवल। ओके एव्रीथिंग सो थिंग्स आर डिसाइडेड। सो यू व्हाट डू दे डू एंड व्हाट डु दीज़ बिग टीम्स डू? दे टेक द बेस्ट प्लेयर्स 18 17 19 20 एंड मेक देमसेल ऑन द बेंच। भाई किसी और के पास नहीं जाने चाहिए रिच। तो वो तीन साल बैठते हैं बेंच पे। द मेन टाइम दे विल इंप्रूव इज बिटवीन दैट टाइम एंड दे डोंट गेट गेम टाइम एंड देन दे गेट रूइन। दैट इज व्हाई न्यू नो न्यू प्लेयर्स हैव कम इंटू 

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [73]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)
vector_store = FAISS.from_documents(chunks, embeddings)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [74]:
vector_store.index_to_docstore_id

{0: '59e1d584-adc0-484d-828f-27e50acdeb03',
 1: '6e16fa7c-0370-4ceb-9fa1-d5ac3e6d9f95',
 2: '013ae10d-5c87-4e7e-b277-1410a0f510bc',
 3: 'b3e6ef82-6a47-42f6-bbe4-4f34a2b6febd',
 4: 'ec968a3a-445e-452e-a617-bcc1ee4b649e',
 5: 'ad7ddeb6-a3bb-4cbb-a3a3-17b33695e98c',
 6: '949af398-070b-4e3a-842f-158f8a3bd6ea',
 7: '33916c73-cce5-4462-aa29-e506564c19da',
 8: '10dd4fb9-fc9a-4864-a459-2b70baf73ce9',
 9: '12da9799-01a0-43bf-ac0d-0d656904280e',
 10: 'b5cdd2dc-5e12-4b2e-9014-bc33bbd633b2',
 11: '63d0e3f2-4232-44c5-97b4-4ae500f35feb',
 12: '5de5b1b8-2ec7-4c2c-90ee-5da17d5fd815',
 13: '7fab76b6-980d-4b44-9b79-7a45bfe86051',
 14: '38640669-eb96-4bc5-9b1c-28ba6b26d10a',
 15: '4050dd2e-160b-4098-9a6b-56d1bfa1f6b9',
 16: '40382d62-f0a7-46a1-b8eb-47c3fdf3e6fb',
 17: 'ca174160-e7b2-4c64-ba61-b44ac5bb671e',
 18: '168401e7-71cd-423d-8dec-3d43f7bcdce4',
 19: '3553246a-f52f-4084-a64a-9b13b9173433',
 20: '05fc57f6-a2f6-4ac3-ae25-b93b066c7c8b',
 21: '0f32afd0-8f4f-4df3-be35-28eb8c2650d3',
 22: 'dbaa6e8e-7db3-

In [75]:
vector_store.get_by_ids(['7f1279e7-9055-44f7-b4c1-138c74db62fe'])

[Document(id='7f1279e7-9055-44f7-b4c1-138c74db62fe', metadata={}, page_content='गंदा सिस्टम है यहां पे वो बंदा माइग्रेट करके चले जाता है वहां पे एंड अंडरस्टैंड मेनी माइग्रेट्स सो इफ ही माइग्रेट्स ना ही इज़ नॉट डूइंग वेल इफ इफ यू आर अ बिलिनियर हियर यू विल नॉट माइग्रेट ना सो इफ यू आर नॉट डूइंग वेल यू वांट बेटर अपोरर्चुनिटीज़ एंड चांसेस फॉर फैमिली यू गो आउट दैट मींस यू डोंट हैव मैन मनी यू डोंट हैव एनी रिसोर्सेज देयर सो यू सेंड योर किड टू द फ्री एकेडमी और समथिंग लाइक दैट वो एक हमारा केरला से गया वो साला कतार से खेल गया ये कतार पपुलेशन 15 1.5 मिलियन अगेन अगेन फिर और वहां पर न्यूजीलैंड की पापुलेशन अगेन वो उसमें से हमारा एक बंदा गया ही इज़ आल्सो मेरिट मैन यहां से गया वो भी मेरिट है तो हमारा सिस्टम कितना गंदा है यार इट्स जस्ट व्हाई आई एम सेइंग इट्स गॉट नथिंग टू डू विद एनीथिंग एल्स एक्सेप्ट द सिस्टम एक्सप्लेन मी अबाउट सिस्टम ओके वी आई विल टॉक सिस्टमैटिक क्वेश्चन ऑन दिस राइट सो एक्चुअली बिफोर वी टॉक अबा अबाउट सिस्टम यू सेड समथिंग अबाउट मेसी वी गव हिम 150 क्रोस नो, द प्राइवेट ऑर्गेनाइजर हु ऑर्गेनाइज

## Step 2 - Retrieval

In [94]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 20})

In [77]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002234CDB2990>, search_kwargs={'k': 7})

In [78]:
retriever.invoke('What is deepmind')

[Document(id='27a775c0-725f-4f6a-a680-458565b7bbe5', metadata={}, page_content='इन द कंट्री राइट नाउ। 4000 800 टर्फ्स तो हमारी उड़ीसा में है। सो हां। सो लेट मी टेल यू अबाउट वन स्टेट हाउ दे चेंज्ड दिस इज़ द नीड टू बी नोन एस अ बिमारू स्टेट फॉर सो एवरीवन बीमारू स्टेट वाज़ नेम देयर वर फाइव सिक्स स्टेट्स बीमारू बेसिकली वाज़ अ फुल फॉर्म ऑफ़ समथिंग दैट दे वर नॉट अ इकोनोमिकली वेल सो बिहार वाज़ वन ऑफ़ देम एंड देन उड़ीसा वाज़ वन ऑफ़ देम। सो दिस चीफ मिनिस्टर सेड ओके आई वांट टेक ऑन माइसेल्फ टू ब्रिंग बैक हॉकी। सो ही मेड स्टेडियम्स। ही हैड द हॉकी वर्ल्ड लीग। ही है ही होस्टेड टू हॉकी होल्ड वर्ल्ड कप्स। एंड ही मेड ओपन हिज़ टाइम ओवर 200, 300 टर्फ्स, ऑल ओवर, एव्री डिस्ट्रिक्ट हैड अ टर्फ। सो, एव्री किड इन एव्री डिस्ट्रिक्ट यूज्ड टू गो एंड प्ले, 50% ऑफ द इंडियन हॉकी टीम इज़ नाउ मेड बाय ट्राइबल्स फ्रॉम झारखंड एंड उड़ीसा एंड द वीमेन। 50% बिकॉज़ ऑफ़ दैट वन मैन। वन मैनस सो हाउ मच वाज़ देयर बजट? व्हेन इंडिया स्पोर्ट बजट वाज़ 1500 करोड़, उड़ीसा स्पोर्ट बजट वाज़ 1300 करोड़। उड़ीसा हो। इमेजिन दैट व्हेन वी हैड एंटायर बजट एंड देन द दे 

## Step 3 - Augmentation

In [79]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    api_key=os.getenv("GROQ_API_KEY")
)

In [80]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [81]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [82]:
retrieved_docs

[Document(id='8d40e4f3-b189-41ab-b17a-f7a47532ba79', metadata={}, page_content='अप बिकॉज़ यू आर आउट ऑफ़ योर कंफर्ट ज़ोन। एंड आई बिलीव यू ओनली ग्रो आउट ऑफ़ कंफर्ट ज़ोन। इफ यू आर सी सपो आ- आ- आ- आ- आ- आ- आ- आ- आ- आ- आई टेल देम। सपो दीज़ कैंपर्स कम फॉर वन ऑर टू मंथ्स। सो स्कूल पीपल डु नो। सो व्हेन दे कम इन दिस से आई से इफ यू आर फीलिंग कम्फर्टेबल हियर इन द फर्स्ट फाइव ओर सिक्स डेज देन आई नॉट डूइंग माय जॉब प्रोपरली यू ओनली ग्रो आउट ऑफ योर कम्फर्ट ज़ोन देट मीन्स यू बेटर बी अनकम्फर्टेबल एंड ओनली देन यू हैव ग्रोथ सो इफ यू थिंक मिरर वाज़ अ प्लेस ऑफ़ लक्ज़री इट्स नॉट एट आल। एंड इट्स अ प्लेस ऑफ़ मेकिंग श्योर दैट आई पुश यू टू योर लिमिट दैट मींस। सो आई टेल माय वॉइस दैट लिसेन। इफ आई स्टॉप अब्यूजिंग यू आर पुशिंग यू और आई एम आफ्टर यू दैट मींस आई हैव गिवन अप ऑन यू। आई एम ओनली पुशिंग यू एंड इफ आई एम रियली सपोज़ आई एम यू आर द वन आई एम पिकिंग ऑन द मोस्ट यू आर द मोस्ट लकीस्ट गाए मैन बिकॉज़ आई रियली थिंक दैट यू कैन गो रियली फार एंड यू नॉट डूइंग जस्टिस टू योर पोटेंशियल एंड माई जॉब इस टू मेक श्योर नो पोटेंशियल टैलेंट गेट्

In [83]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'अप बिकॉज़ यू आर आउट ऑफ़ योर कंफर्ट ज़ोन। एंड आई बिलीव यू ओनली ग्रो आउट ऑफ़ कंफर्ट ज़ोन। इफ यू आर सी सपो आ- आ- आ- आ- आ- आ- आ- आ- आ- आ- आई टेल देम। सपो दीज़ कैंपर्स कम फॉर वन ऑर टू मंथ्स। सो स्कूल पीपल डु नो। सो व्हेन दे कम इन दिस से आई से इफ यू आर फीलिंग कम्फर्टेबल हियर इन द फर्स्ट फाइव ओर सिक्स डेज देन आई नॉट डूइंग माय जॉब प्रोपरली यू ओनली ग्रो आउट ऑफ योर कम्फर्ट ज़ोन देट मीन्स यू बेटर बी अनकम्फर्टेबल एंड ओनली देन यू हैव ग्रोथ सो इफ यू थिंक मिरर वाज़ अ प्लेस ऑफ़ लक्ज़री इट्स नॉट एट आल। एंड इट्स अ प्लेस ऑफ़ मेकिंग श्योर दैट आई पुश यू टू योर लिमिट दैट मींस। सो आई टेल माय वॉइस दैट लिसेन। इफ आई स्टॉप अब्यूजिंग यू आर पुशिंग यू और आई एम आफ्टर यू दैट मींस आई हैव गिवन अप ऑन यू। आई एम ओनली पुशिंग यू एंड इफ आई एम रियली सपोज़ आई एम यू आर द वन आई एम पिकिंग ऑन द मोस्ट यू आर द मोस्ट लकीस्ट गाए मैन बिकॉज़ आई रियली थिंक दैट यू कैन गो रियली फार एंड यू नॉट डूइंग जस्टिस टू योर पोटेंशियल एंड माई जॉब इस टू मेक श्योर नो पोटेंशियल टैलेंट गेट्स नचरಡ್ नो पोटेंशियल गेट्स वेस्टेड सो आई एम गोना पुश योर एस एंड इफ यू\n\nआर लाइ

In [84]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [85]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      अप बिकॉज़ यू आर आउट ऑफ़ योर कंफर्ट ज़ोन। एंड आई बिलीव यू ओनली ग्रो आउट ऑफ़ कंफर्ट ज़ोन। इफ यू आर सी सपो आ- आ- आ- आ- आ- आ- आ- आ- आ- आ- आई टेल देम। सपो दीज़ कैंपर्स कम फॉर वन ऑर टू मंथ्स। सो स्कूल पीपल डु नो। सो व्हेन दे कम इन दिस से आई से इफ यू आर फीलिंग कम्फर्टेबल हियर इन द फर्स्ट फाइव ओर सिक्स डेज देन आई नॉट डूइंग माय जॉब प्रोपरली यू ओनली ग्रो आउट ऑफ योर कम्फर्ट ज़ोन देट मीन्स यू बेटर बी अनकम्फर्टेबल एंड ओनली देन यू हैव ग्रोथ सो इफ यू थिंक मिरर वाज़ अ प्लेस ऑफ़ लक्ज़री इट्स नॉट एट आल। एंड इट्स अ प्लेस ऑफ़ मेकिंग श्योर दैट आई पुश यू टू योर लिमिट दैट मींस। सो आई टेल माय वॉइस दैट लिसेन। इफ आई स्टॉप अब्यूजिंग यू आर पुशिंग यू और आई एम आफ्टर यू दैट मींस आई हैव गिवन अप ऑन यू। आई एम ओनली पुशिंग यू एंड इफ आई एम रियली सपोज़ आई एम यू आर द वन आई एम पिकिंग ऑन द मोस्ट यू आर द मोस्ट लकीस्ट गाए मैन बिकॉज़ आई रियली थिंक दैट यू

## Step 4 - Generation

In [86]:
answer = llm.invoke(final_prompt)
print(answer.content)

No, the topic of nuclear fusion is not discussed in this video. The conversation appears to be about football, personal growth, and motivation, with some discussion about the Indian football system and the challenges it faces.


## Building a Chain

In [87]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [88]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [89]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [90]:
parallel_chain.invoke('who is Demis')

{'context': 'बिफोर योर ब्रेन ऑलरेडी थिंक दैट यू हैव डन इट। योर ब्रेन कैन बी रिव सी द ब्रेन इज सो इजी टू फूल। इट कैन बी रिवायर्ड इन एनीवेयर यू वांट। यू कैन गो इंटू डीप स्पाइरल। व्हाई बिकॉज़ यू कैन से दैट ओके आई एम डिप्रेस्ड। एंड योर ब्रेन विल थिंक दैट्स ट्रू। यू आर डिप्रेस्ड, एंड यू विल कीप ऑन रिवायरिंग इट, एंड दोज़ कनेक्शंस विल कीप बिकमिंग स्ट्रोंगर। द सेम वे इफ यू कीप टेलिंग योरसेल्फ यू द बेस्ट इन द वर्ल्ड, यू डोंट बिकम द बेस्ट इन द वर्ल्ड, पीपल अंडरस्टैंड दैट। बट व्हाट हप्पेंस इज यू स्टार्ट बिलीविंग इन योरसेल्फ। द मोमेंट यू स्टार्ट बिलीविंग इन योरसेल्फ, दैट्स व्हेन यू स्टार्ट हैविंग मोर कॉन्फिडेंस। योर कॉन्फिडेंस कम्स फ्रॉम द अमाउंट ऑफ वर्क यू पुट इन। पीपल से ओह माय गॉड, कॉन्फिडेंस नहीं। कॉन्फिडेंस नहीं है बिकॉज़ योर बॉडी नोज़? बिकॉज़ हु कांट यू लाइक टू योरसेल्फ। यू कांट लाइट योरसेल्फ, एंड यू नो योर यू क्राैप। इफ यू आर द वर्ल्ड नंबर वन, यू विल बी कॉन्फिडेंट, व्हाई? बिकॉज़ आई एम वर्ल्ड नंबर वन ऑर आई एम दैट गुड इन माय सपोर्ट। सो व्हाई डू थिंक आई एम दैट कॉन्फिडेंट व्हेनवर आई गो एंड आई से आई ग

In [91]:
parser = StrOutputParser()

In [92]:
main_chain = parallel_chain | prompt | llm | parser

In [95]:
main_chain.invoke('Summarize the video')

'इस वीडियो में एक व्यक्ति अपने अनुभव और विचार साझा कर रहा है, जो खेल, प्रेरणा, और जीवन के विभिन्न पहलुओं पर केंद्रित है। वह अपने खेल जीवन, खासकर फुटबॉल में अपनी उपलब्धियों और चुनौतियों के बारे में बात करता है, और बताता है कि कैसे उसने अपने लक्ष्यों को प्राप्त करने के लिए कड़ी मेहनत की और अपने आप पर विश्वास रखा।\n\nवह यह भी बताता है कि कैसे उसने अपने खिलाड़ियों को प्रेरित किया और उन्हें अपने लक्ष्यों को प्राप्त करने के लिए प्रोत्साहित किया। वह अपने खेल जीवन में आने वाली चुनौतियों और उनसे निपटने के तरीकों के बारे में भी बात करता है।\n\nवह अपने देश, भारत में प्रतिभा को बढ़ावा देने और खेलों में उत्कृष्टता प्राप्त करने की आवश्यकता पर भी जोर देता है। वह कहता है कि भारत में प्रतिभा को बढ़ावा देने के लिए एक अच्छा प्रणाली होनी चाहिए, जो खिलाड़ियों को अपने लक्ष्यों को प्राप्त करने में मदद कर सके।\n\nवह अपने जीवन के अनुभवों और खेल जीवन में अपनी उपलब्धियों के बारे में भी बात करता है, और बताता है कि कैसे उसने अपने लक्ष्यों को प्राप्त करने के लिए कड़ी मेहनत की और अपने आप पर विश्वास रखा। वह अपने जीवन